# 07: Final Expected LOS Model Selection and Scoring

## Purpose

This notebook uses the results from Notebook 06 to select and save the final
expected LOS model for this project.

The notebook:

- Checks XGBoost against the model selection criteria
- Saves the final feature list and model settings
- Creates hospital-held-out predictions for the 2023 data
- Fits the selected model on all eligible records and saves the model file
- Records the model version and scoring time
- Creates the `ModelPrediction` table for Power BI
- Summarizes prediction coverage and reasons for missing predictions
- Checks row counts, keys, values, fold separation, and model reload results
- Exports the supporting tables and model documentation

## Why Cross-Fitted Scores Are Used

A model evaluated on the same records used for training can look more accurate
than it really is. This is a concern for hospital comparisons because the model
would already have seen records from the hospital being evaluated.

For the Power BI table, each hospital is scored by a model trained without that
hospital. A separate model fitted on all eligible records is saved for later
data, subject to schema and stability checks.

## Scope

This is a retrospective expected LOS model based on the primary features defined
in Notebook 06. It is not an admission time clinical model. Its results do not
measure hospital quality or prove that excess LOS is preventable.

`peer_expected_los_days` and `model_predicted_los_days` remain separate analytical concepts.


## 1. Imports


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import platform
import time

import duckdb
import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)


## 2. Project Paths and Version Configuration


In [2]:
def find_project_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "docs" / "project_charter.md").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Expected docs/project_charter.md "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
AUDIT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
CATALOG_DIR = PROJECT_ROOT / "outputs" / "metric_catalog"
SCHEMA_DIR = PROJECT_ROOT / "outputs" / "star_schema"
PHYSICAL_DIR = PROJECT_ROOT / "outputs" / "physical_model"
PHYSICAL_TABLE_DIR = PHYSICAL_DIR / "tables"
BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "peer_benchmarks"
BENCHMARK_TABLE_DIR = BENCHMARK_DIR / "tables"
DEVELOPMENT_DIR = PROJECT_ROOT / "outputs" / "expected_los_model"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "expected_los_scoring"
TABLE_DIR = OUTPUT_DIR / "tables"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
WORK_DIR = OUTPUT_DIR / "_work"
DOCS_DIR = PROJECT_ROOT / "docs"
WORK_DB_PATH = WORK_DIR / "final_expected_los_scoring.duckdb"

for directory in [OUTPUT_DIR, TABLE_DIR, ARTIFACT_DIR, WORK_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SELECTED_CANDIDATE = "XGBOOST"
STRONG_BASELINE = "APR_DRG_SEVERITY_MEAN"
MODEL_VERSION = "expected_los_xgb_2023_v1.0.0"
SCORING_TIMESTAMP = datetime.now(timezone.utc).replace(microsecond=0)

RANDOM_SEED = 42
N_GROUP_FOLDS = 3
ONE_HOT_MIN_FREQUENCY = 25
PREDICTION_FLOOR_DAYS = 0.01
XGBOOST_TREES = 350
REPORTING_MIN_N = 11

environment_versions = pd.DataFrame([
    ["python", platform.python_version()],
    ["pandas", pd.__version__],
    ["numpy", np.__version__],
    ["duckdb", duckdb.__version__],
    ["scikit-learn", sklearn.__version__],
    ["xgboost", xgboost.__version__],
    ["joblib", joblib.__version__],
], columns=["component", "version"])

print("Project root:", PROJECT_ROOT)
print("Model version:", MODEL_VERSION)
print("Scoring timestamp (UTC):", SCORING_TIMESTAMP.isoformat())
display(environment_versions)


Project root: C:\Users\kdy10\OneDrive\Documents\ChatGPT\Repo Revisit\04_hospital_operations_powerbi
Model version: expected_los_xgb_2023_v1.0.0
Scoring timestamp (UTC): 2026-08-31T09:01:16+00:00


,component,version
0,python,3.14.3
1,pandas,3.0.5
2,numpy,2.5.1
3,duckdb,1.5.5
4,scikit-learn,1.9.0
5,xgboost,3.4.1
6,joblib,1.5.3


## 3. Load Upstream Contracts and Validation Results


In [3]:
input_files = {
    "file_metadata": AUDIT_DIR / "file_metadata.csv",
    "catalog_validation": CATALOG_DIR / "validation_results.csv",
    "schema_validation": SCHEMA_DIR / "schema_validation_results.csv",
    "physical_validation": PHYSICAL_DIR / "physical_validation_results.csv",
    "physical_parquet_validation": PHYSICAL_DIR / "parquet_validation.csv",
    "benchmark_validation": BENCHMARK_DIR / "benchmark_validation_results.csv",
    "benchmark_parquet_validation": BENCHMARK_DIR / "parquet_validation.csv",
    "modeling_validation": DEVELOPMENT_DIR / "modeling_validation_results.csv",
    "model_comparison": DEVELOPMENT_DIR / "model_comparison.csv",
    "model_shortlist": DEVELOPMENT_DIR / "model_shortlist.csv",
    "feature_specification": DEVELOPMENT_DIR / "feature_specification.csv",
    "prediction_output_specification": DEVELOPMENT_DIR / "prediction_output_specification.csv",
    "subgroup_performance": DEVELOPMENT_DIR / "subgroup_performance.csv",
    "development_environment": DEVELOPMENT_DIR / "environment_versions.csv",
}

table_paths = {
    "FactDischarge": BENCHMARK_TABLE_DIR / "FactDischarge.parquet",
    "DimHospital": PHYSICAL_TABLE_DIR / "DimHospital.parquet",
    "DimService": PHYSICAL_TABLE_DIR / "DimService.parquet",
    "DimCaseMix": PHYSICAL_TABLE_DIR / "DimCaseMix.parquet",
    "DimDiagnosis": PHYSICAL_TABLE_DIR / "DimDiagnosis.parquet",
    "DimPatientSegment": PHYSICAL_TABLE_DIR / "DimPatientSegment.parquet",
    "DimPayer": PHYSICAL_TABLE_DIR / "DimPayer.parquet",
    "DimAdmissionContext": PHYSICAL_TABLE_DIR / "DimAdmissionContext.parquet",
}

missing_files = [
    str(path)
    for path in [*input_files.values(), *table_paths.values()]
    if not path.exists()
]
assert not missing_files, "Required upstream outputs are missing:\n" + "\n".join(missing_files)

inputs = {name: pd.read_csv(path) for name, path in input_files.items()}
file_metadata = inputs["file_metadata"]
model_comparison = inputs["model_comparison"]
model_shortlist = inputs["model_shortlist"]
feature_specification = inputs["feature_specification"]
prediction_output_specification = inputs["prediction_output_specification"]
subgroup_performance = inputs["subgroup_performance"]

assert len(file_metadata) == 1
SOURCE_SHA256 = str(file_metadata.loc[0, "sha256"]).strip()
SOURCE_SNAPSHOT_ID = SOURCE_SHA256[:16]

display(model_shortlist)
print("Source snapshot ID:", SOURCE_SNAPSHOT_ID)


,shortlist_rank,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs,model_name,candidate_type,total_cv_seconds,mae_rank,median_absolute_error_rank,rmse_rank,calibration_error_abs_rank,multi_metric_rank_score,mae_improvement_vs_strong_baseline_pct
0,1,250000,3.284980,1.658697,6.959015,5.782,5.776028,1445500.0,1.444007e+06,1.001034,0.001034,XGBOOST,Statistical / ML,14.506381,2.0,4.0,1.0,7.0,3.5,1.550821
1,2,250000,3.424587,1.734176,7.128632,5.782,5.783009,1445500.0,1.445752e+06,0.999825,0.000175,RIDGE_REGRESSION,Statistical / ML,5.939483,5.0,5.0,3.0,3.0,4.0,-2.633142
2,3,250000,3.512404,2.082932,7.361639,5.782,5.782320,1445500.0,1.445580e+06,0.999945,0.000055,RANDOM_FOREST,Statistical / ML,10.151408,6.0,7.0,5.0,2.0,5.0,-5.264961


Source snapshot ID: d69e4b9e47fd2992


## 4. Upstream Commit Gates


In [4]:
def normalize_boolean(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )


upstream_frames = {
    "Notebook 02 metric catalog": inputs["catalog_validation"],
    "Notebook 03 schema": inputs["schema_validation"],
    "Notebook 04 physical model": inputs["physical_validation"],
    "Notebook 04 Parquet": inputs["physical_parquet_validation"],
    "Notebook 05 benchmark": inputs["benchmark_validation"],
    "Notebook 05 Parquet": inputs["benchmark_parquet_validation"],
    "Notebook 06 modeling": inputs["modeling_validation"],
}

upstream_gate_rows = []
for label, frame in upstream_frames.items():
    passed = normalize_boolean(frame["passed"]).fillna(False).all()
    upstream_gate_rows.append([label, bool(passed)])

upstream_commit_gates = pd.DataFrame(
    upstream_gate_rows,
    columns=["upstream_artifact", "passed"],
)
display(upstream_commit_gates)
assert upstream_commit_gates["passed"].all(), "An upstream validation gate failed."


,upstream_artifact,passed
0,Notebook 02 metric catalog,True
1,Notebook 03 schema,True
2,Notebook 04 physical model,True
3,Notebook 04 Parquet,True
4,Notebook 05 benchmark,True
5,Notebook 05 Parquet,True
6,Notebook 06 modeling,True


## 5. Promotion Policy

The selection does not rely on MAE alone. XGBoost must meet each of the
following checks:

- Improve MAE by at least 1% compared with the APR-DRG by severity mean
- Keep RMSE no worse than the same baseline
- Keep aggregate calibration error within 0.01
- Limit median absolute error deterioration to 5%
- Improve MAE for at least 60% of reportable hospitals
- Complete the development run within 120 seconds
- Use no fields that Notebook 06 marked as forbidden

These thresholds apply to this project. They are not clinical model acceptance
standards.


In [5]:
comparison = model_comparison.set_index("model_name")
candidate = comparison.loc[SELECTED_CANDIDATE]
baseline = comparison.loc[STRONG_BASELINE]

hospital_subgroups = subgroup_performance.loc[
    subgroup_performance["subgroup_dimension"].eq("hospital_key")
    & subgroup_performance["model_name"].isin([SELECTED_CANDIDATE, STRONG_BASELINE])
    & subgroup_performance["n"].ge(REPORTING_MIN_N)
].copy()

hospital_pivot = hospital_subgroups.pivot(
    index="subgroup_value",
    columns="model_name",
    values=["n", "mae", "actual_to_expected_ratio"],
).dropna()

hospital_mae_win_share = float(
    (hospital_pivot[("mae", SELECTED_CANDIDATE)]
     < hospital_pivot[("mae", STRONG_BASELINE)]).mean()
)
candidate_hospital_median_calibration_error = float(
    (hospital_pivot[("actual_to_expected_ratio", SELECTED_CANDIDATE)] - 1).abs().median()
)
baseline_hospital_median_calibration_error = float(
    (hospital_pivot[("actual_to_expected_ratio", STRONG_BASELINE)] - 1).abs().median()
)

primary_features = feature_specification.loc[
    normalize_boolean(feature_specification["included_primary"]).fillna(False),
    "feature_name",
].tolist()
forbidden_features = set(feature_specification.loc[
    feature_specification["role"].eq("Forbidden primary"),
    "feature_name",
])

promotion_policy = pd.DataFrame([
    ["MAE improvement vs strong baseline", ">= 1.0%", float(candidate["mae_improvement_vs_strong_baseline_pct"]), float(candidate["mae_improvement_vs_strong_baseline_pct"]) >= 1.0,
     "Requires a measurable hospital-held-out improvement over the transparent conditional-mean baseline."],
    ["RMSE non-inferiority", "candidate <= baseline", float(candidate["rmse"]), float(candidate["rmse"]) <= float(baseline["rmse"]),
     f"Baseline RMSE = {float(baseline['rmse']):.6f}."],
    ["Aggregate calibration error", "<= 0.01", float(candidate["calibration_error_abs"]), float(candidate["calibration_error_abs"]) <= 0.01,
     "Keeps aggregate actual-to-expected calibration within one percentage point."],
    ["Median absolute error degradation", "<= 5.0%", (float(candidate["median_absolute_error"]) / float(baseline["median_absolute_error"]) - 1) * 100,
     (float(candidate["median_absolute_error"]) / float(baseline["median_absolute_error"]) - 1) * 100 <= 5.0,
     "Prevents a small mean-error gain from masking a material typical-case deterioration."],
    ["Hospital subgroup MAE win share", ">= 0.60", hospital_mae_win_share, hospital_mae_win_share >= 0.60,
     "Uses only Notebook 06 hospital subgroup rows meeting the 11-discharge reporting threshold."],
    ["Hospital median calibration non-inferiority", "candidate-baseline <= 0.02",
     candidate_hospital_median_calibration_error - baseline_hospital_median_calibration_error,
     candidate_hospital_median_calibration_error - baseline_hospital_median_calibration_error <= 0.02,
     "Limits deterioration in the median hospital absolute A/E deviation."],
    ["Development runtime", "<= 120 seconds", float(candidate["total_cv_seconds"]), float(candidate["total_cv_seconds"]) <= 120,
     "Confirms local refit and scoring remain operationally practical."],
    ["Primary feature contract", "no forbidden overlap", float(len(set(primary_features) & forbidden_features)), not bool(set(primary_features) & forbidden_features),
     "Protects the leakage and analytical-objective boundary defined in Notebook 06."],
], columns=["promotion_gate", "threshold", "observed_value", "passed", "rationale"])

display(promotion_policy)
assert promotion_policy["passed"].all(), "The provisional candidate did not pass every promotion gate."


,promotion_gate,threshold,observed_value,passed,rationale
0,MAE improvement vs strong baseline,>= 1.0%,1.550821,True,Requires a measurable hospital-held-out improvement over the transparent conditional-mean baseline.
1,RMSE non-inferiority,candidate <= baseline,6.959015,True,Baseline RMSE = 7.061390.
2,Aggregate calibration error,<= 0.01,0.001034,True,Keeps aggregate actual-to-expected calibration within one percentage point.
3,Median absolute error degradation,<= 5.0%,1.509214,True,Prevents a small mean-error gain from masking a material typical-case deterioration.
4,Hospital subgroup MAE win share,>= 0.60,0.700980,True,Uses only Notebook 06 hospital subgroup rows meeting the 11-discharge reporting threshold.
5,Hospital median calibration non-inferiority,candidate-baseline <= 0.02,-0.003074,True,Limits deterioration in the median hospital absolute A/E deviation.
6,Development runtime,<= 120 seconds,14.506381,True,Confirms local refit and scoring remain operationally practical.
7,Primary feature contract,no forbidden overlap,0.000000,True,Protects the leakage and analytical-objective boundary defined in Notebook 06.


## 6. Final Model Selection


In [6]:
final_model_selection = pd.DataFrame([
    ["Selected model", SELECTED_CANDIDATE, "Promoted after passing every multi-criterion gate."],
    ["Model version", MODEL_VERSION, "Frozen semantic version for the audited 2023 source contract."],
    ["Strong transparent comparator", STRONG_BASELINE, "Retained as a benchmark and monitoring guardrail."],
    ["Hospital-held-out MAE", f"{float(candidate['mae']):.6f}", "Notebook 06 out-of-fold result."],
    ["MAE improvement vs comparator", f"{float(candidate['mae_improvement_vs_strong_baseline_pct']):.4f}%", "Incremental rather than transformative improvement."],
    ["Aggregate actual-to-expected", f"{float(candidate['actual_to_expected_ratio']):.6f}", "Notebook 06 development calibration."],
    ["Current-snapshot scoring method", "Hospital-held-out cross-fitting", "Avoids using in-sample predictions for hospital comparison."],
    ["Future compatible scoring artifact", "Full eligible-population refit", "Stored separately for controlled later scoring; future-year validity is not established."],
], columns=["decision_item", "value", "interpretation"])

display(final_model_selection)


,decision_item,value,interpretation
0,Selected model,XGBOOST,Promoted after passing every multi-criterion gate.
1,Model version,expected_los_xgb_2023_v1.0.0,Frozen semantic version for the audited 2023 source contract.
2,Strong transparent comparator,APR_DRG_SEVERITY_MEAN,Retained as a benchmark and monitoring guardrail.
3,Hospital-held-out MAE,3.284980,Notebook 06 out-of-fold result.
4,MAE improvement vs comparator,1.5508%,Incremental rather than transformative improvement.
5,Aggregate actual-to-expected,1.001034,Notebook 06 development calibration.
6,Current-snapshot scoring method,Hospital-held-out cross-fitting,Avoids using in-sample predictions for hospital comparison.
7,Future compatible scoring artifact,Full eligible-population refit,Stored separately for controlled later scoring; future-year validity is not established.


### Selection Interpretation

XGBoost passes every selection check. In the Notebook 06 development sample,
its MAE was 1.55% lower than the APR-DRG by severity mean. It also had a lower
RMSE, close overall calibration, and a lower MAE for 70.1% of reportable
hospitals.

The improvement is small. The leave-one-facility-out peer benchmark remains a
separate descriptive measure in Power BI and is not replaced by this model.


## 7. Freeze the Feature and Hyperparameter Contracts


In [7]:
EXPECTED_PRIMARY_FEATURES = [
    "apr_drg_code",
    "apr_mdc_code",
    "medical_surgical_classification",
    "apr_severity_code",
    "apr_mortality_risk",
    "ccsr_diagnosis_code",
    "age_group",
    "gender",
    "payer_group",
    "admission_type_group",
    "ed_indicator_group",
]
assert primary_features == EXPECTED_PRIMARY_FEATURES, (
    "Notebook 06 primary feature order or membership changed. Review before refitting."
)

final_feature_contract = feature_specification.copy()
final_feature_contract.insert(0, "model_version", MODEL_VERSION)
final_feature_contract["contract_status"] = np.where(
    normalize_boolean(final_feature_contract["included_primary"]).fillna(False),
    "FROZEN_PRIMARY",
    final_feature_contract["role"].str.upper().str.replace(" ", "_"),
)

final_model_configuration = pd.DataFrame([
    ["model_family", "XGBRegressor"],
    ["objective", "reg:squarederror"],
    ["n_estimators", XGBOOST_TREES],
    ["learning_rate", 0.05],
    ["max_depth", 8],
    ["min_child_weight", 10],
    ["subsample", 0.80],
    ["colsample_bytree", 0.80],
    ["reg_lambda", 1.0],
    ["tree_method", "hist"],
    ["eval_metric", "mae"],
    ["random_state", RANDOM_SEED],
    ["one_hot_min_frequency", ONE_HOT_MIN_FREQUENCY],
    ["prediction_floor_days", PREDICTION_FLOOR_DAYS],
    ["current_snapshot_scoring", "3-fold hospital-held-out cross-fitting"],
    ["full_refit_population", "All eligible rows in audited source snapshot"],
], columns=["parameter", "value"])
final_model_configuration.insert(0, "model_version", MODEL_VERSION)

display(final_feature_contract)
display(final_model_configuration)


,model_version,feature_name,source_table,role,included_primary,rationale,contract_status
0,expected_los_xgb_2023_v1.0.0,apr_drg_code,DimService,Primary feature,True,Core discharge clinical/service classification.,FROZEN_PRIMARY
1,expected_los_xgb_2023_v1.0.0,apr_mdc_code,DimService,Primary feature,True,Broad diagnostic-category context.,FROZEN_PRIMARY
2,expected_los_xgb_2023_v1.0.0,medical_surgical_classification,DimService,Primary feature,True,Medical-versus-surgical context.,FROZEN_PRIMARY
3,expected_los_xgb_2023_v1.0.0,apr_severity_code,DimCaseMix,Primary feature,True,Governed severity-of-illness case-mix measure.,FROZEN_PRIMARY
4,expected_los_xgb_2023_v1.0.0,apr_mortality_risk,DimCaseMix,Primary feature,True,Released mortality-risk case-complexity context.,FROZEN_PRIMARY
5,expected_los_xgb_2023_v1.0.0,ccsr_diagnosis_code,DimDiagnosis,Primary feature,True,Diagnosis-category case-mix context.,FROZEN_PRIMARY
6,expected_los_xgb_2023_v1.0.0,age_group,DimPatientSegment,Primary feature,True,Approved age case-mix context.,FROZEN_PRIMARY
7,expected_los_xgb_2023_v1.0.0,gender,DimPatientSegment,Primary feature,True,Released demographic context.,FROZEN_PRIMARY
8,expected_los_xgb_2023_v1.0.0,payer_group,DimPayer,Primary feature,True,Approved payer-mix context.,FROZEN_PRIMARY
9,expected_los_xgb_2023_v1.0.0,admission_type_group,DimAdmissionContext,Primary feature,True,Admission context.,FROZEN_PRIMARY


,model_version,parameter,value
0,expected_los_xgb_2023_v1.0.0,model_family,XGBRegressor
1,expected_los_xgb_2023_v1.0.0,objective,reg:squarederror
2,expected_los_xgb_2023_v1.0.0,n_estimators,350
3,expected_los_xgb_2023_v1.0.0,learning_rate,0.05
4,expected_los_xgb_2023_v1.0.0,max_depth,8
5,expected_los_xgb_2023_v1.0.0,min_child_weight,10
6,expected_los_xgb_2023_v1.0.0,subsample,0.8
7,expected_los_xgb_2023_v1.0.0,colsample_bytree,0.8
8,expected_los_xgb_2023_v1.0.0,reg_lambda,1.0
9,expected_los_xgb_2023_v1.0.0,tree_method,hist


## 8. Load the Governed Analytical Tables


In [8]:
def sql_path_literal(path):
    return "'" + Path(path).resolve().as_posix().replace("'", "''") + "'"


def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()

benchmark_fact_hash_before = sha256_file(table_paths["FactDischarge"])
con = duckdb.connect(database=str(WORK_DB_PATH))

for table_name, table_path in table_paths.items():
    con.execute(
        f"""CREATE OR REPLACE VIEW {quote_identifier(table_name + "Input")} AS
            SELECT * FROM read_parquet({sql_path_literal(table_path)})"""
    )

fact_row_count = int(con.sql(
    'SELECT COUNT(*) AS row_count FROM "FactDischargeInput"'
).df().loc[0, "row_count"])

print("Benchmarked FactDischarge rows:", f"{fact_row_count:,}")
print("Input SHA-256:", benchmark_fact_hash_before)


Benchmarked FactDischarge rows: 2,125,754
Input SHA-256: fcc01282307aee3bdf2f30c01c75c0ec565408e36e223ddcaf2284cae159e8c8


## 9. Build the Final Modeling Source


In [9]:
modeling_df = con.sql("""
    SELECT
        f.source_record_key,
        f.hospital_key,
        svc.apr_drg_code,
        svc.apr_mdc_code,
        svc.medical_surgical_classification,
        cm.apr_severity_code,
        cm.apr_mortality_risk,
        dx.ccsr_diagnosis_code,
        ps.age_group,
        ps.gender,
        pay.payer_group,
        ac.admission_type_group,
        ac.ed_indicator_group,
        f.los_days_lower_bound,
        f.is_top_coded_los,
        f.is_valid_los
    FROM "FactDischargeInput" AS f
    LEFT JOIN "DimServiceInput" AS svc ON f.service_key = svc.service_key
    LEFT JOIN "DimCaseMixInput" AS cm ON f.case_mix_key = cm.case_mix_key
    LEFT JOIN "DimDiagnosisInput" AS dx ON f.diagnosis_key = dx.diagnosis_key
    LEFT JOIN "DimPatientSegmentInput" AS ps ON f.patient_segment_key = ps.patient_segment_key
    LEFT JOIN "DimPayerInput" AS pay ON f.payer_key = pay.payer_key
    LEFT JOIN "DimAdmissionContextInput" AS ac ON f.admission_context_key = ac.admission_context_key
    ORDER BY f.source_record_key
""").df()

assert len(modeling_df) == fact_row_count
assert modeling_df["source_record_key"].notna().all()
assert modeling_df["source_record_key"].is_unique

for feature in primary_features:
    modeling_df[feature] = (
        modeling_df[feature]
        .astype("string")
        .fillna("Unknown / Not Available")
        .astype("category")
    )

target = pd.to_numeric(modeling_df["los_days_lower_bound"], errors="coerce").to_numpy(float)
eligible_mask = (
    modeling_df["is_valid_los"].eq(1).to_numpy()
    & np.isfinite(target)
    & (target > 0)
    & modeling_df["hospital_key"].ne(0).to_numpy()
)
eligible_positions = np.flatnonzero(eligible_mask)
eligible_n = int(eligible_mask.sum())
eligible_hospital_n = int(modeling_df.loc[eligible_mask, "hospital_key"].nunique())

scoring_population_summary = pd.DataFrame([{
    "source_snapshot_id": SOURCE_SNAPSHOT_ID,
    "fact_rows": fact_row_count,
    "eligible_rows": eligible_n,
    "ineligible_rows": fact_row_count - eligible_n,
    "eligible_hospitals": eligible_hospital_n,
    "eligible_top_coded_rows": int(modeling_df.loc[eligible_mask, "is_top_coded_los"].eq(1).sum()),
    "eligible_mean_los_lower_bound": float(np.mean(target[eligible_mask])),
    "eligible_median_los_lower_bound": float(np.median(target[eligible_mask])),
}])

display(scoring_population_summary)


,source_snapshot_id,fact_rows,eligible_rows,ineligible_rows,eligible_hospitals,eligible_top_coded_rows,eligible_mean_los_lower_bound,eligible_median_los_lower_bound
0,d69e4b9e47fd2992,2125754,2120421,5333,207,2272,5.785011,3.0


## 10. Final Pipeline Definition


In [10]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=ONE_HOT_MIN_FREQUENCY,
            sparse_output=True,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=ONE_HOT_MIN_FREQUENCY,
            sparse=True,
            dtype=np.float32,
        )


def make_final_pipeline():
    preprocessor = ColumnTransformer(
        [("categorical", make_one_hot_encoder(), primary_features)],
        remainder="drop",
        sparse_threshold=1.0,
    )
    estimator = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=XGBOOST_TREES,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=10,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="mae",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", estimator)])


def clip_predictions(values):
    return np.maximum(np.asarray(values, dtype=float), PREDICTION_FLOOR_DAYS)


def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = clip_predictions(y_pred)
    actual_sum = float(np.sum(y_true))
    predicted_sum = float(np.sum(y_pred))
    return {
        "n": len(y_true),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "median_absolute_error": float(median_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "actual_mean": float(np.mean(y_true)),
        "predicted_mean": float(np.mean(y_pred)),
        "actual_sum": actual_sum,
        "predicted_sum": predicted_sum,
        "actual_to_expected_ratio": actual_sum / predicted_sum,
        "calibration_error_abs": abs(actual_sum / predicted_sum - 1.0),
    }


## 11. Hospital-Held-Out Cross-Fitted Scoring


In [11]:
features = modeling_df[primary_features]
groups = modeling_df.loc[eligible_mask, "hospital_key"].to_numpy()
group_cv = GroupKFold(n_splits=N_GROUP_FOLDS)

crossfit_predictions = np.full(fact_row_count, np.nan, dtype=np.float32)
crossfit_fold = np.zeros(fact_row_count, dtype=np.int8)
fold_metric_rows = []
fold_isolation_rows = []

for fold_number, (train_local, validation_local) in enumerate(
    group_cv.split(eligible_positions, groups=groups), start=1
):
    train_positions = eligible_positions[train_local]
    validation_positions = eligible_positions[validation_local]
    train_hospitals = set(modeling_df.iloc[train_positions]["hospital_key"].astype(int))
    validation_hospitals = set(modeling_df.iloc[validation_positions]["hospital_key"].astype(int))
    hospital_overlap_n = len(train_hospitals & validation_hospitals)

    model = make_final_pipeline()
    started = time.perf_counter()
    model.fit(features.iloc[train_positions], target[train_positions])
    fold_predictions = clip_predictions(model.predict(features.iloc[validation_positions]))
    elapsed = time.perf_counter() - started

    crossfit_predictions[validation_positions] = fold_predictions.astype(np.float32)
    crossfit_fold[validation_positions] = fold_number

    metrics = evaluate_predictions(target[validation_positions], fold_predictions)
    metrics.update({
        "fold": fold_number,
        "training_rows": len(train_positions),
        "validation_rows": len(validation_positions),
        "training_hospitals": len(train_hospitals),
        "validation_hospitals": len(validation_hospitals),
        "fit_predict_seconds": elapsed,
    })
    fold_metric_rows.append(metrics)
    fold_isolation_rows.append({
        "fold": fold_number,
        "training_hospitals": len(train_hospitals),
        "validation_hospitals": len(validation_hospitals),
        "hospital_overlap_n": hospital_overlap_n,
        "passed": hospital_overlap_n == 0,
    })

    print(
        f"Fold {fold_number}: validation rows={len(validation_positions):,}; "
        f"MAE={metrics['mae']:.4f}; A/E={metrics['actual_to_expected_ratio']:.4f}; "
        f"seconds={elapsed:.1f}"
    )
    del model, fold_predictions
    gc.collect()

crossfit_fold_metrics = pd.DataFrame(fold_metric_rows)
fold_isolation_validation = pd.DataFrame(fold_isolation_rows)

assert np.isfinite(crossfit_predictions[eligible_mask]).all()
assert np.isnan(crossfit_predictions[~eligible_mask]).all()
assert (crossfit_fold[eligible_mask] > 0).all()
assert fold_isolation_validation["passed"].all()

display(crossfit_fold_metrics)
display(fold_isolation_validation)


Fold 1: validation rows=706,797; MAE=2.9568; A/E=0.9564; seconds=44.7


Fold 2: validation rows=706,792; MAE=3.4811; A/E=1.0298; seconds=49.9


Fold 3: validation rows=706,832; MAE=3.3220; A/E=1.0147; seconds=52.7


,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs,fold,training_rows,validation_rows,training_hospitals,validation_hospitals,fit_predict_seconds
0,706797,2.956811,1.596587,5.997649,5.300864,5.542323,3746635.0,3.917297e+06,0.956434,0.043566,1,1413624,706797,138,69,44.737410
1,706792,3.481068,1.719170,7.545854,6.189680,6.010820,4374816.0,4.248399e+06,1.029756,0.029756,2,1413629,706792,138,69,49.878060
2,706832,3.321960,1.666060,7.087473,5.864487,5.779393,4145207.0,4.085060e+06,1.014724,0.014724,3,1413589,706832,138,69,52.719958


,fold,training_hospitals,validation_hospitals,hospital_overlap_n,passed
0,1,138,69,0,True
1,2,138,69,0,True
2,3,138,69,0,True


## 12. Current-Snapshot Cross-Fitted Performance


In [12]:
full_population_crossfit_performance = pd.DataFrame([
    {
        "model_version": MODEL_VERSION,
        "scoring_method": "3-fold hospital-held-out cross-fitting",
        **evaluate_predictions(target[eligible_mask], crossfit_predictions[eligible_mask]),
    }
])

development_candidate_metrics = pd.DataFrame([{
    "model_version": MODEL_VERSION,
    "evaluation_population": "Notebook 06 deterministic 250,000-row development sample",
    "mae": float(candidate["mae"]),
    "median_absolute_error": float(candidate["median_absolute_error"]),
    "rmse": float(candidate["rmse"]),
    "actual_to_expected_ratio": float(candidate["actual_to_expected_ratio"]),
}])

display(full_population_crossfit_performance)
display(development_candidate_metrics)


,model_version,scoring_method,n,mae,median_absolute_error,rmse,actual_mean,predicted_mean,actual_sum,predicted_sum,actual_to_expected_ratio,calibration_error_abs
0,expected_los_xgb_2023_v1.0.0,3-fold hospital-held-out cross-fitting,2120421,3.25328,1.658492,6.907581,5.785011,5.777512,12266658.0,1.225076e+07,1.001298,0.001298


,model_version,evaluation_population,mae,median_absolute_error,rmse,actual_to_expected_ratio
0,expected_los_xgb_2023_v1.0.0,"Notebook 06 deterministic 250,000-row development sample",3.28498,1.658697,6.959015,1.001034


## 13. Refit the Frozen Model on All Eligible Rows


In [13]:
final_model = make_final_pipeline()
final_fit_started = time.perf_counter()
final_model.fit(features.iloc[eligible_positions], target[eligible_positions])
final_fit_seconds = time.perf_counter() - final_fit_started

MODEL_ARTIFACT_PATH = ARTIFACT_DIR / f"{MODEL_VERSION}.joblib"
if MODEL_ARTIFACT_PATH.exists():
    MODEL_ARTIFACT_PATH.unlink()
joblib.dump(final_model, MODEL_ARTIFACT_PATH, compress=3)

assert MODEL_ARTIFACT_PATH.exists()
assert MODEL_ARTIFACT_PATH.stat().st_size > 0

model_artifact_sha256 = sha256_file(MODEL_ARTIFACT_PATH)
model_artifact_metadata = pd.DataFrame([{
    "model_version": MODEL_VERSION,
    "model_family": "XGBRegressor",
    "source_snapshot_id": SOURCE_SNAPSHOT_ID,
    "training_rows": eligible_n,
    "training_hospitals": eligible_hospital_n,
    "fit_seconds": final_fit_seconds,
    "artifact_file": MODEL_ARTIFACT_PATH.name,
    "artifact_sha256": model_artifact_sha256,
    "artifact_size_mb": round(MODEL_ARTIFACT_PATH.stat().st_size / (1024 ** 2), 2),
    "created_at_utc": SCORING_TIMESTAMP.isoformat(),
}])

display(model_artifact_metadata)


,model_version,model_family,source_snapshot_id,training_rows,training_hospitals,fit_seconds,artifact_file,artifact_sha256,artifact_size_mb,created_at_utc
0,expected_los_xgb_2023_v1.0.0,XGBRegressor,d69e4b9e47fd2992,2120421,207,74.923843,expected_los_xgb_2023_v1.0.0.joblib,5e427925da85eba8204d7af47fdb0c72e5c573f6308045c50e1355425f76f16a,0.88,2026-08-31T09:01:16+00:00


## 14. Serialized-Artifact Smoke Test


In [14]:
smoke_positions = eligible_positions[: min(1000, eligible_n)]
predictions_before_reload = clip_predictions(final_model.predict(features.iloc[smoke_positions]))
reloaded_model = joblib.load(MODEL_ARTIFACT_PATH)
predictions_after_reload = clip_predictions(reloaded_model.predict(features.iloc[smoke_positions]))

artifact_smoke_test = pd.DataFrame([{
    "sample_rows": len(smoke_positions),
    "max_absolute_prediction_difference": float(
        np.max(np.abs(predictions_before_reload - predictions_after_reload))
    ),
    "all_predictions_positive": bool((predictions_after_reload > 0).all()),
    "passed": bool(
        np.allclose(predictions_before_reload, predictions_after_reload, rtol=0, atol=1e-10)
        and (predictions_after_reload > 0).all()
    ),
}])

display(artifact_smoke_test)
assert artifact_smoke_test["passed"].all()

del reloaded_model, predictions_before_reload, predictions_after_reload
gc.collect()


,sample_rows,max_absolute_prediction_difference,all_predictions_positive,passed
0,1000,0.0,True,True


183

## 15. Build the Versioned Power BI Prediction Table


In [15]:
prediction_values = pd.DataFrame({
    "source_record_key": modeling_df.loc[eligible_mask, "source_record_key"].astype("int64").to_numpy(),
    "model_predicted_los_days": crossfit_predictions[eligible_mask].astype(float),
    "crossfit_fold": crossfit_fold[eligible_mask].astype(int),
})
con.register("prediction_values", prediction_values)

source_snapshot_sql = SOURCE_SNAPSHOT_ID.replace("'", "''")
model_version_sql = MODEL_VERSION.replace("'", "''")
scoring_timestamp_sql = SCORING_TIMESTAMP.isoformat().replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE TABLE ModelPredictionExport AS
    SELECT
        CAST(f.source_record_key AS BIGINT) AS source_record_key,
        '{source_snapshot_sql}'::VARCHAR AS source_snapshot_id,
        '{model_version_sql}'::VARCHAR AS model_version,
        TIMESTAMPTZ '{scoring_timestamp_sql}' AS scoring_timestamp,
        CAST(p.model_predicted_los_days AS DOUBLE) AS model_predicted_los_days,
        CASE WHEN p.source_record_key IS NOT NULL THEN 'SCORED_CROSSFIT' ELSE 'NOT_SCORED' END::VARCHAR AS prediction_status,
        CASE
            WHEN p.source_record_key IS NOT NULL THEN ''
            WHEN f.is_valid_los <> 1 OR f.los_days_lower_bound IS NULL THEN 'INVALID_OR_MISSING_LOS'
            WHEN f.hospital_key = 0 THEN 'UNRESOLVED_HOSPITAL'
            ELSE 'UNEXPECTED_ELIGIBILITY_FAILURE'
        END::VARCHAR AS missing_prediction_reason,
        CAST(p.crossfit_fold AS TINYINT) AS crossfit_fold
    FROM "FactDischargeInput" AS f
    LEFT JOIN prediction_values AS p
        ON f.source_record_key = p.source_record_key
    ORDER BY f.source_record_key
""")

PREDICTION_TABLE_PATH = TABLE_DIR / "ModelPrediction.parquet"
if PREDICTION_TABLE_PATH.exists():
    PREDICTION_TABLE_PATH.unlink()

con.execute(f"""
    COPY ModelPredictionExport
    TO {sql_path_literal(PREDICTION_TABLE_PATH)}
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

assert PREDICTION_TABLE_PATH.exists()
assert PREDICTION_TABLE_PATH.stat().st_size > 0
print("Prediction table:", PREDICTION_TABLE_PATH.relative_to(PROJECT_ROOT))


Prediction table: outputs\expected_los_scoring\tables\ModelPrediction.parquet


## 16. Prediction Coverage and Distribution


In [16]:
scoring_coverage = con.sql("""
    SELECT
        prediction_status,
        missing_prediction_reason,
        COUNT(*) AS row_n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS row_pct
    FROM ModelPredictionExport
    GROUP BY prediction_status, missing_prediction_reason
    ORDER BY prediction_status, missing_prediction_reason
""").df()

prediction_distribution = con.sql("""
    SELECT
        COUNT(model_predicted_los_days) AS scored_n,
        MIN(model_predicted_los_days) AS min_predicted_los,
        QUANTILE_CONT(model_predicted_los_days, 0.25) AS p25_predicted_los,
        MEDIAN(model_predicted_los_days) AS median_predicted_los,
        AVG(model_predicted_los_days) AS mean_predicted_los,
        QUANTILE_CONT(model_predicted_los_days, 0.75) AS p75_predicted_los,
        QUANTILE_CONT(model_predicted_los_days, 0.95) AS p95_predicted_los,
        MAX(model_predicted_los_days) AS max_predicted_los
    FROM ModelPredictionExport
    WHERE prediction_status = 'SCORED_CROSSFIT'
""").df()

display(scoring_coverage)
display(prediction_distribution)


,prediction_status,missing_prediction_reason,row_n,row_pct
0,NOT_SCORED,UNRESOLVED_HOSPITAL,5333,0.2509
1,SCORED_CROSSFIT,,2120421,99.7491


,scored_n,min_predicted_los,p25_predicted_los,median_predicted_los,mean_predicted_los,p75_predicted_los,p95_predicted_los,max_predicted_los
0,2120421,0.01,2.713443,4.164906,5.777512,6.735304,14.236323,113.288635


## 17. Power BI Integration Contract


In [17]:
power_bi_integration_specification = pd.DataFrame([
    ["Table", "ModelPrediction", "Load outputs/expected_los_scoring/tables/ModelPrediction.parquet."],
    ["Grain", "One row per source_record_key for one approved model version", "The current artifact covers every FactDischarge row and contains one model version."],
    ["Join key", "source_record_key", "Snapshot-scoped technical row key; not a patient or durable discharge identifier."],
    ["Relationship", "FactDischarge 1:1 ModelPrediction", "Use an active relationship for the single approved version, or merge the approved prediction columns into FactDischarge in Power Query."],
    ["Version filter", MODEL_VERSION, "Do not mix predictions from multiple model versions in one expected-LOS measure."],
    ["Scored-row filter", "prediction_status = SCORED_CROSSFIT", "Actual-to-model-expected measures must use the same scored population in numerator and denominator."],
    ["Expected LOS", "SUM(model_predicted_los_days)", "Aggregate raw-day conditional-mean predictions only across scored rows."],
    ["Actual LOS", "SUM(FactDischarge[los_days_lower_bound]) on scored rows", "Use the same prediction-eligible population as expected LOS."],
    ["Actual-to-Expected", "Actual LOS / Model-Expected LOS", "Suppress or return blank when expected LOS is zero or unavailable."],
    ["Excess LOS", "Actual LOS - Model-Expected LOS", "Lower-bound interpretation remains because 120+ LOS is top-coded."],
    ["Peer benchmark", "Retain separately", "Do not overwrite or relabel peer_expected_los_days as a model prediction."],
    ["Small-cell reporting", "Minimum displayed n = 11", "Implement the governed suppression policy before presenting filtered model results."],
], columns=["integration_item", "specification", "implementation_note"])

display(power_bi_integration_specification)


,integration_item,specification,implementation_note
0,Table,ModelPrediction,Load outputs/expected_los_scoring/tables/ModelPrediction.parquet.
1,Grain,One row per source_record_key for one approved model version,The current artifact covers every FactDischarge row and contains one model version.
2,Join key,source_record_key,Snapshot-scoped technical row key; not a patient or durable discharge identifier.
3,Relationship,FactDischarge 1:1 ModelPrediction,"Use an active relationship for the single approved version, or merge the approved prediction columns into FactDischarge in Power Query."
4,Version filter,expected_los_xgb_2023_v1.0.0,Do not mix predictions from multiple model versions in one expected-LOS measure.
5,Scored-row filter,prediction_status = SCORED_CROSSFIT,Actual-to-model-expected measures must use the same scored population in numerator and denominator.
6,Expected LOS,SUM(model_predicted_los_days),Aggregate raw-day conditional-mean predictions only across scored rows.
7,Actual LOS,SUM(FactDischarge[los_days_lower_bound]) on scored rows,Use the same prediction-eligible population as expected LOS.
8,Actual-to-Expected,Actual LOS / Model-Expected LOS,Suppress or return blank when expected LOS is zero or unavailable.
9,Excess LOS,Actual LOS - Model-Expected LOS,Lower-bound interpretation remains because 120+ LOS is top-coded.


## 18. Validate the Exported Prediction Artifact


In [18]:
exported_schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet({sql_path_literal(PREDICTION_TABLE_PATH)})
""").df()
exported_profile = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(source_record_key) AS nonnull_key_n,
        COUNT(DISTINCT source_record_key) AS distinct_key_n,
        SUM(CASE WHEN prediction_status = 'SCORED_CROSSFIT' THEN 1 ELSE 0 END) AS scored_n,
        SUM(CASE WHEN prediction_status = 'SCORED_CROSSFIT' AND model_predicted_los_days IS NULL THEN 1 ELSE 0 END) AS scored_missing_prediction_n,
        SUM(CASE WHEN prediction_status = 'SCORED_CROSSFIT' AND model_predicted_los_days <= 0 THEN 1 ELSE 0 END) AS scored_nonpositive_prediction_n,
        SUM(CASE WHEN prediction_status = 'NOT_SCORED' AND model_predicted_los_days IS NOT NULL THEN 1 ELSE 0 END) AS unscored_nonnull_prediction_n,
        SUM(CASE WHEN prediction_status = 'NOT_SCORED' AND missing_prediction_reason = '' THEN 1 ELSE 0 END) AS unscored_blank_reason_n,
        COUNT(DISTINCT model_version) AS model_version_n,
        COUNT(DISTINCT source_snapshot_id) AS source_snapshot_n
    FROM read_parquet({sql_path_literal(PREDICTION_TABLE_PATH)})
""").df()

required_contract_columns = set(
    prediction_output_specification.loc[
        normalize_boolean(prediction_output_specification["required"]).fillna(False),
        "column_name",
    ]
)
exported_columns = set(exported_schema["column_name"])
profile = exported_profile.iloc[0]

benchmark_fact_hash_after = sha256_file(table_paths["FactDischarge"])

scoring_validation_results = pd.DataFrame([
    ["All upstream validation gates passed", upstream_commit_gates["passed"].all(), ""],
    ["All promotion gates passed", promotion_policy["passed"].all(), SELECTED_CANDIDATE],
    ["Frozen primary feature contract contains no forbidden fields", not bool(set(primary_features) & forbidden_features), str(len(primary_features))],
    ["Modeling source preserves fact row count", len(modeling_df) == fact_row_count, f"{len(modeling_df)} vs {fact_row_count}"],
    ["Every eligible row received one cross-fitted prediction", np.isfinite(crossfit_predictions[eligible_mask]).all(), str(eligible_n)],
    ["No ineligible row received a cross-fitted prediction", np.isnan(crossfit_predictions[~eligible_mask]).all(), str(fact_row_count - eligible_n)],
    ["Cross-fitting has zero hospital overlap", fold_isolation_validation["passed"].all(), str(int(fold_isolation_validation["hospital_overlap_n"].sum()))],
    ["Serialized model reload reproduces predictions", artifact_smoke_test["passed"].all(), model_artifact_sha256],
    ["Export preserves fact row count", int(profile["row_count"]) == fact_row_count, f"{int(profile['row_count'])} vs {fact_row_count}"],
    ["Exported source_record_key is complete and unique", int(profile["row_count"]) == int(profile["nonnull_key_n"]) == int(profile["distinct_key_n"]), str(int(profile["distinct_key_n"]))],
    ["Exported scored count matches eligible population", int(profile["scored_n"]) == eligible_n, f"{int(profile['scored_n'])} vs {eligible_n}"],
    ["Scored predictions are complete", int(profile["scored_missing_prediction_n"]) == 0, str(int(profile["scored_missing_prediction_n"]))],
    ["Scored predictions are strictly positive", int(profile["scored_nonpositive_prediction_n"]) == 0, str(int(profile["scored_nonpositive_prediction_n"]))],
    ["Unscored rows have null predictions", int(profile["unscored_nonnull_prediction_n"]) == 0, str(int(profile["unscored_nonnull_prediction_n"]))],
    ["Unscored rows have a reason", int(profile["unscored_blank_reason_n"]) == 0, str(int(profile["unscored_blank_reason_n"]))],
    ["Required Notebook 06 prediction columns are present", required_contract_columns.issubset(exported_columns), "|".join(sorted(required_contract_columns))],
    ["Export contains one model version", int(profile["model_version_n"]) == 1, str(int(profile["model_version_n"]))],
    ["Export contains one source snapshot", int(profile["source_snapshot_n"]) == 1, str(int(profile["source_snapshot_n"]))],
    ["Benchmarked fact input remained immutable", benchmark_fact_hash_before == benchmark_fact_hash_after, benchmark_fact_hash_after],
], columns=["validation_test", "passed", "details"])

display(exported_schema)
display(exported_profile)
display(scoring_validation_results)
assert scoring_validation_results["passed"].all(), "Notebook 07 failed a scoring validation gate."


,column_name,column_type,null,key,default,extra
0,source_record_key,BIGINT,YES,None,None,None
1,source_snapshot_id,VARCHAR,YES,None,None,None
2,model_version,VARCHAR,YES,None,None,None
3,scoring_timestamp,TIMESTAMP WITH TIME ZONE,YES,None,None,None
4,model_predicted_los_days,DOUBLE,YES,None,None,None
5,prediction_status,VARCHAR,YES,None,None,None
6,missing_prediction_reason,VARCHAR,YES,None,None,None
7,crossfit_fold,TINYINT,YES,None,None,None


,row_count,nonnull_key_n,distinct_key_n,scored_n,scored_missing_prediction_n,scored_nonpositive_prediction_n,unscored_nonnull_prediction_n,unscored_blank_reason_n,model_version_n,source_snapshot_n
0,2125754,2125754,2125754,2120421.0,0.0,0.0,0.0,0.0,1,1


,validation_test,passed,details
0,All upstream validation gates passed,True,
1,All promotion gates passed,True,XGBOOST
2,Frozen primary feature contract contains no forbidden fields,True,11
3,Modeling source preserves fact row count,True,2125754 vs 2125754
4,Every eligible row received one cross-fitted prediction,True,2120421
5,No ineligible row received a cross-fitted prediction,True,5333
6,Cross-fitting has zero hospital overlap,True,0
7,Serialized model reload reproduces predictions,True,5e427925da85eba8204d7af47fdb0c72e5c573f6308045c50e1355425f76f16a
8,Export preserves fact row count,True,2125754 vs 2125754
9,Exported source_record_key is complete and unique,True,2125754


## 19. Export Governance, Validation, and Diagnostic Artifacts


In [19]:
export_frames = {
    "environment_versions.csv": environment_versions,
    "upstream_commit_gates.csv": upstream_commit_gates,
    "promotion_policy.csv": promotion_policy,
    "final_model_selection.csv": final_model_selection,
    "final_feature_contract.csv": final_feature_contract,
    "final_model_configuration.csv": final_model_configuration,
    "scoring_population_summary.csv": scoring_population_summary,
    "crossfit_fold_metrics.csv": crossfit_fold_metrics,
    "fold_isolation_validation.csv": fold_isolation_validation,
    "full_population_crossfit_performance.csv": full_population_crossfit_performance,
    "development_candidate_metrics.csv": development_candidate_metrics,
    "model_artifact_metadata.csv": model_artifact_metadata,
    "artifact_smoke_test.csv": artifact_smoke_test,
    "scoring_coverage.csv": scoring_coverage,
    "prediction_distribution.csv": prediction_distribution,
    "power_bi_integration_specification.csv": power_bi_integration_specification,
    "scoring_validation_results.csv": scoring_validation_results,
}

manifest_rows = []
for file_name, frame in export_frames.items():
    output_path = OUTPUT_DIR / file_name
    frame.to_csv(output_path, index=False)
    assert output_path.exists() and output_path.stat().st_size > 0
    manifest_rows.append({
        "file_name": file_name,
        "artifact_type": "Governance, validation, or diagnostic output",
        "row_count": len(frame),
        "file_size_mb": round(output_path.stat().st_size / (1024 ** 2), 4),
        "output_path": output_path.relative_to(PROJECT_ROOT).as_posix(),
    })

manifest_rows.extend([
    {
        "file_name": PREDICTION_TABLE_PATH.name,
        "artifact_type": "Power BI-ready versioned prediction table",
        "row_count": fact_row_count,
        "file_size_mb": round(PREDICTION_TABLE_PATH.stat().st_size / (1024 ** 2), 2),
        "output_path": PREDICTION_TABLE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
    {
        "file_name": MODEL_ARTIFACT_PATH.name,
        "artifact_type": "Full eligible-population fitted model artifact",
        "row_count": pd.NA,
        "file_size_mb": round(MODEL_ARTIFACT_PATH.stat().st_size / (1024 ** 2), 2),
        "output_path": MODEL_ARTIFACT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
])

export_manifest = pd.DataFrame(manifest_rows)
display(export_manifest)


,file_name,artifact_type,row_count,file_size_mb,output_path
0,environment_versions.csv,"Governance, validation, or diagnostic output",7,0.0001,outputs/expected_los_scoring/environment_versions.csv
1,upstream_commit_gates.csv,"Governance, validation, or diagnostic output",7,0.0002,outputs/expected_los_scoring/upstream_commit_gates.csv
2,promotion_policy.csv,"Governance, validation, or diagnostic output",8,0.0011,outputs/expected_los_scoring/promotion_policy.csv
3,final_model_selection.csv,"Governance, validation, or diagnostic output",8,0.0008,outputs/expected_los_scoring/final_model_selection.csv
4,final_feature_contract.csv,"Governance, validation, or diagnostic output",31,0.0046,outputs/expected_los_scoring/final_feature_contract.csv
5,final_model_configuration.csv,"Governance, validation, or diagnostic output",16,0.0009,outputs/expected_los_scoring/final_model_configuration.csv
6,scoring_population_summary.csv,"Governance, validation, or diagnostic output",1,0.0002,outputs/expected_los_scoring/scoring_population_summary.csv
7,crossfit_fold_metrics.csv,"Governance, validation, or diagnostic output",3,0.0008,outputs/expected_los_scoring/crossfit_fold_metrics.csv
8,fold_isolation_validation.csv,"Governance, validation, or diagnostic output",3,0.0001,outputs/expected_los_scoring/fold_isolation_validation.csv
9,full_population_crossfit_performance.csv,"Governance, validation, or diagnostic output",1,0.0004,outputs/expected_los_scoring/full_population_crossfit_performance.csv


## 20. Generate `docs/final_expected_los_model.md`


In [20]:
def dataframe_to_markdown(dataframe, columns):
    selected = dataframe.loc[:, columns].fillna("").astype(str)
    def escape(value):
        return value.replace("|", r"\|").replace("\n", " ")
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = [
        "| " + " | ".join(escape(value) for value in row) + " |"
        for row in selected.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])


selection_md = dataframe_to_markdown(
    final_model_selection, ["decision_item", "value", "interpretation"]
)
promotion_md = dataframe_to_markdown(
    promotion_policy, ["promotion_gate", "threshold", "observed_value", "passed"]
)
coverage_md = dataframe_to_markdown(
    scoring_coverage, ["prediction_status", "missing_prediction_reason", "row_n", "row_pct"]
)
performance_md = dataframe_to_markdown(
    full_population_crossfit_performance,
    ["n", "mae", "median_absolute_error", "rmse", "actual_to_expected_ratio"],
)
validation_md = dataframe_to_markdown(
    scoring_validation_results, ["validation_test", "passed", "details"]
)

final_model_document = "\n".join([
    "# Final Expected-LOS Model and Versioned Scoring",
    "",
    "## Purpose",
    "",
    "This document records the final development-stage expected-LOS model decision, the frozen feature and parameter contracts, current-snapshot cross-fitted scoring, the full-data refit artifact, and the Power BI handoff.",
    "",
    "## Final Selection",
    "",
    selection_md,
    "",
    "## Promotion Gates",
    "",
    promotion_md,
    "",
    "XGBoost is promoted with a modest improvement over the transparent APR-DRG × severity conditional-mean baseline. The transparent leave-one-facility-out peer benchmark remains a separate explanatory and diagnostic measure.",
    "",
    "## Current-Snapshot Scoring Design",
    "",
    "The audited 2023 source is scored with three-fold hospital-held-out cross-fitting. Every scored hospital is evaluated by a fold model trained without that hospital. This prevents the Power BI artifact from using ordinary in-sample predictions for current-snapshot hospital comparison.",
    "",
    "A separate full eligible-population model artifact is retained for controlled scoring of later schema-compatible data. The current one-year project does not demonstrate future-year stability.",
    "",
    "## Full-Population Cross-Fitted Performance",
    "",
    performance_md,
    "",
    "## Prediction Coverage",
    "",
    coverage_md,
    "",
    "## Power BI Artifact",
    "",
    f"- Prediction table: `{PREDICTION_TABLE_PATH.relative_to(PROJECT_ROOT).as_posix()}`",
    f"- Model artifact: `{MODEL_ARTIFACT_PATH.relative_to(PROJECT_ROOT).as_posix()}`",
    f"- Model version: `{MODEL_VERSION}`",
    f"- Source snapshot: `{SOURCE_SNAPSHOT_ID}`",
    "- Join key: `source_record_key`",
    "- Use only `prediction_status = SCORED_CROSSFIT` when calculating actual-to-model-expected measures.",
    "- Apply the governed minimum-11 reporting control before displaying filtered results.",
    "",
    "## Validation Results",
    "",
    validation_md,
    "",
    "## Interpretation Boundaries",
    "",
    "- The target is the observable LOS lower bound; `120 +` remains right-censored.",
    "- Predictions are retrospective operational expectations, not admission-time clinical forecasts.",
    "- Expected differences do not prove causality, preventability, inefficiency, or poor quality.",
    "- Patient disposition, hospital identity, procedures, charges, costs, peer expectations, and LOS-derived fields are not primary model features.",
    "- `model_predicted_los_days` does not replace `peer_expected_los_days`.",
    "- Multi-year compatibility, temporal stability, and production monitoring remain future work.",
    "",
])

FINAL_MODEL_DOCUMENT_PATH = DOCS_DIR / "final_expected_los_model.md"
FINAL_MODEL_DOCUMENT_PATH.write_text(final_model_document, encoding="utf-8")
assert FINAL_MODEL_DOCUMENT_PATH.exists() and FINAL_MODEL_DOCUMENT_PATH.stat().st_size > 0
print("Documentation:", FINAL_MODEL_DOCUMENT_PATH.relative_to(PROJECT_ROOT))


Documentation: docs\final_expected_los_model.md


## 21. Finalize the Export Manifest


In [21]:
documentation_row = pd.DataFrame([{
    "file_name": FINAL_MODEL_DOCUMENT_PATH.name,
    "artifact_type": "Documentation",
    "row_count": pd.NA,
    "file_size_mb": round(FINAL_MODEL_DOCUMENT_PATH.stat().st_size / (1024 ** 2), 4),
    "output_path": FINAL_MODEL_DOCUMENT_PATH.relative_to(PROJECT_ROOT).as_posix(),
}])

final_export_manifest = pd.concat(
    [export_manifest, documentation_row], ignore_index=True
)
EXPORT_MANIFEST_PATH = OUTPUT_DIR / "export_manifest.csv"
final_export_manifest.to_csv(EXPORT_MANIFEST_PATH, index=False)

for relative_path in final_export_manifest["output_path"]:
    assert (PROJECT_ROOT / relative_path).exists()

display(final_export_manifest)


,file_name,artifact_type,row_count,file_size_mb,output_path
0,environment_versions.csv,"Governance, validation, or diagnostic output",7,0.0001,outputs/expected_los_scoring/environment_versions.csv
1,upstream_commit_gates.csv,"Governance, validation, or diagnostic output",7,0.0002,outputs/expected_los_scoring/upstream_commit_gates.csv
2,promotion_policy.csv,"Governance, validation, or diagnostic output",8,0.0011,outputs/expected_los_scoring/promotion_policy.csv
3,final_model_selection.csv,"Governance, validation, or diagnostic output",8,0.0008,outputs/expected_los_scoring/final_model_selection.csv
4,final_feature_contract.csv,"Governance, validation, or diagnostic output",31,0.0046,outputs/expected_los_scoring/final_feature_contract.csv
5,final_model_configuration.csv,"Governance, validation, or diagnostic output",16,0.0009,outputs/expected_los_scoring/final_model_configuration.csv
6,scoring_population_summary.csv,"Governance, validation, or diagnostic output",1,0.0002,outputs/expected_los_scoring/scoring_population_summary.csv
7,crossfit_fold_metrics.csv,"Governance, validation, or diagnostic output",3,0.0008,outputs/expected_los_scoring/crossfit_fold_metrics.csv
8,fold_isolation_validation.csv,"Governance, validation, or diagnostic output",3,0.0001,outputs/expected_los_scoring/fold_isolation_validation.csv
9,full_population_crossfit_performance.csv,"Governance, validation, or diagnostic output",1,0.0004,outputs/expected_los_scoring/full_population_crossfit_performance.csv


## 22. Clean Temporary Build Artifacts


In [22]:
con.close()

if WORK_DB_PATH.exists():
    WORK_DB_PATH.unlink()
try:
    WORK_DIR.rmdir()
except OSError:
    pass

print("Temporary DuckDB scoring artifacts removed.")
print("Notebook 07 outputs retained successfully.")


Temporary DuckDB scoring artifacts removed.
Notebook 07 outputs retained successfully.


# Final Notebook Summary

## Work Completed

- Loaded the validated outputs from Notebooks 02 through 06.
- Compared XGBoost with the APR-DRG by severity mean baseline.
- Selected XGBoost after it passed every selection check.
- Saved the final feature list and XGBoost settings.
- Kept forbidden and sensitivity fields out of the primary feature list.
- Built the eligible modeling population from the star schema.
- Created three-fold hospital-held-out predictions for the 2023 data.
- Confirmed that no hospital appeared in both sides of a fold.
- Fitted the selected model on all eligible records.
- Saved, hashed, reloaded, and tested the fitted model file.
- Created `ModelPrediction.parquet` with one row for every fact record.
- Added scoring status and missing reason fields.
- Confirmed that the benchmarked fact table did not change.
- Checked the exported row count, keys, coverage, values, model version, and source snapshot.
- Exported the model selection, scoring, validation, and Power BI support files.
- Generated `docs/final_expected_los_model.md`.

## Final Decision

`expected_los_xgb_2023_v1.0.0` is the selected model version for retrospective
analysis of the 2023 data.

Power BI uses hospital-held-out predictions for the 2023 data. The model fitted
on all eligible records is saved for later compatible data, but it should not
be used until schema and stability checks pass.

## Remaining Work

- Implement DAX measures using matched scored populations.
- Add small cell suppression in Power BI.
- Build and validate the Power BI semantic model and report pages.
- Reconcile Power BI measures against independent Python calculations.
- Test the model on a later year before considering production use.
- Define monitoring for drift, calibration, subgroup results, and model versions.
- Design the Power BI Desktop and Fabric deployment process.

## Scope Boundary

This notebook creates a versioned retrospective expected LOS model for
operational analysis. It does not support causal claims, hospital quality
ratings, clinical recommendations, or admission time predictions.
